In [13]:
from ultralytics import YOLO
import cv2
from datetime import datetime

In [14]:
model = YOLO('runs/train/product_model/weights/best.pt')
current_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
img = "images/val/IMG_4664.jpeg"

In [15]:
results = model.predict(source=img, conf=0.5, save = True, save_txt = True, name=f'product_predict_{current_time}') # predict bounding boxes
# results = model.predict(source=img, conf=0.5)


image 1/1 /Users/andreanoh/Documents/Spring 2025 6010 ComputerVision/cv_product_info_extract/images/val/IMG_4664.jpeg: 640x480 1 quarter, 1 product, 138.5ms
Speed: 13.6ms preprocess, 138.5ms inference, 24.0ms postprocess per image at shape (1, 3, 640, 480)
Results saved to /Users/andreanoh/Documents/Spring 2025 6010 ComputerVision/cv_product_info_extract/runs/detect/product_predict_2025-04-12_04-42-30
1 label saved to /Users/andreanoh/Documents/Spring 2025 6010 ComputerVision/cv_product_info_extract/runs/detect/product_predict_2025-04-12_04-42-30/labels


In [16]:
print(results[0].boxes.xyxy) # bounding box coordinates

tensor([[ 853.3783,  728.9912, 2090.2656, 3517.5332],
        [2114.1101, 2159.0369, 2619.1985, 2677.1926]])


In [17]:
print(results[0].boxes.conf) # confidence scores

tensor([0.9338, 0.7715])


In [18]:
for c in results[0].boxes.cls:
    print(model.names[int(c)]) # predicted classes

product
quarter


In [19]:
quarter_diameter = 0.955 # size of a quarter in inches

# extract bounding boxes and class indices from the results (convert tensors to numpy arrays)
bboxes = results[0].boxes.xyxy.cpu().numpy()  # shape: (num_boxes, 4)
classes = results[0].boxes.cls.cpu().numpy()    # class indices for each box

quarter_bbox = None
product_bbox = None

for bbox, cls in zip(bboxes, classes):
    # get the class name from the model's dictionary
    label = model.names[int(cls)].lower()
    if label == "quarter":
        quarter_bbox = bbox
    elif label == "product":
        product_bbox = bbox

if quarter_bbox is None or product_bbox is None:
    raise ValueError("Both quarter and product bounding boxes must be detected.")

# compute the quarter's pixel diameter
q_width = quarter_bbox[2] - quarter_bbox[0]
q_height = quarter_bbox[3] - quarter_bbox[1]
quarter_pixel_diameter = (q_width + q_height) / 2.0

pixel_to_inch = quarter_diameter / quarter_pixel_diameter # conversion factor: inches per pixel

# get product dimensions in pixels
p_width = product_bbox[2] - product_bbox[0]
p_height = product_bbox[3] - product_bbox[1]

# convert product dimensions from pixels to inches
product_width_in = p_width * pixel_to_inch
product_height_in = p_height * pixel_to_inch

print(f"Product dimensions: width = {product_width_in:.2f} inches, height = {product_height_in:.2f} inches")

Product dimensions: width = 2.31 inches, height = 5.21 inches
